# Week 6 실습

> "신청자 평균 소득이 이상하게 높게 나와요. 고소득자 몇 명 때문인 것 같은데 빼고 다시 계산해 주세요. 이상한 값도 다 지워주시고요." — 여신심사팀장

오늘 실습은 팀장 요청대로 이상치를 지워 보고, 그게 맞는 처리인지 확인하는 것이다. 튀는 값을 **오류 / 코드값 / 절단 / 진짜 극단값 / 통계적으로만 드문 값**으로 나누고, 지우면 보고서 숫자가 어떻게 바뀌는지 보는 것이 목표임.

| Part | 내용 | 셀 |
|---|---|---|
| 0 | 도입: 끝에 몰린 막대 (California Housing) | 예측, 실행, 해석 |
| 1 | 데이터와 분포 확인 | 실행, 해석 |
| 2 | 코드값을 먼저 처리한다 (`DAYS_EMPLOYED` 365243) | 예측, 빈칸, 해석 |
| 3 | 기준이 다르면 이상치 개수가 다르다 | 예측, 빈칸, 해석 |
| 4 | 지우기 전에 행을 열어본다 | 예측, 빈칸, 해석 |
| 5 | 지우면 결론이 바뀐다 | 예측, 빈칸, 해석 |
| 6 | 분류표 완성, AI와 비교 | 빈칸, 해석 |

셀 표시
- [실행] 실행만 하는 셀
- [빈칸] `TODO`를 채우는 셀
- [예측] 코드를 돌리기 전에 예상을 적는 칸. 틀린 예측도 기록해 둔다.
- [해석] 결과를 보고 해석을 적는 칸
- [심화] 시간이 부족하면 건너뛰어도 되는 셀

LLM 사용 규칙: 문법과 에러는 자유롭게 사용 가능. 대신에, 방법 선택과 결과 해석은 [해석] 칸에 먼저 적은 뒤에만 사용. AI 비교는 Part 6에서 한 번에 진행할 예정

## 기본 세팅 [실행]

In [ ]:
# 확인할 것: OK (307511, 122)이 출력되는지. 다르면 data/ 경로와 파일(application_train.csv)을 확인한다.
import sys
sys.path.append("../..")   # weekNN_주제/<폴더>/ 기준 레포 루트 (README "처음 세팅" 참고)

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
from IPython.display import display

from common.load_data import load_home_credit
from common.home_credit.preprocess import fix_days_employed

pd.set_option("display.float_format", "{:,.3f}".format)
pd.set_option("display.max_columns", 50)
plt.rcParams["figure.dpi"] = 100

# 그래프 한글 폰트: 설치된 한글 폰트 중 먼저 찾은 것을 쓴다 (Mac: AppleGothic, Windows: 맑은 고딕)
from matplotlib import font_manager
_installed = {f.name for f in font_manager.fontManager.ttflist}
for _font in ["AppleGothic", "Malgun Gothic", "NanumGothic", "Nanum Gothic", "Noto Sans CJK KR", "Noto Sans KR", "Noto Sans CJK JP"]:
    if _font in _installed:
        plt.rcParams["font.family"] = _font
        break
else:
    print("한글 폰트를 찾지 못했어요. 그래프의 한글이 네모로 보이면 나눔고딕을 설치한 뒤 커널을 다시 시작하세요.")
plt.rcParams["axes.unicode_minus"] = False   # 한글 폰트에서 마이너스(-) 기호가 깨지지 않게
BLUE, ORANGE, GRAY = "#2a78d6", "#eb6834", "#9aa5b1"

raw = load_home_credit()   # data/application_train.csv, shape (307511, 122) 확인. Part 2에서 처리 전후 비교에만 쓴다
CODE = 365243              # DAYS_EMPLOYED 코드값 (5주차 D-005)

# 5주차 결정대로 365243을 먼저 처리한다. fix_days_employed()가 아직 main에 없으면 같은 처리(플래그 + NaN)를 임시로 한다.
try:
    _tmp = raw.copy()
    _out = fix_days_employed(_tmp)
    df = _tmp if _out is None else _out
    how = "fix_days_employed()"
except NotImplementedError:
    df = raw.copy()
    df["DAYS_EMPLOYED_ANOM"] = (df["DAYS_EMPLOYED"] == CODE).astype(int)
    df["DAYS_EMPLOYED"] = df["DAYS_EMPLOYED"].replace(CODE, np.nan)
    how = "임시 처리 (fix_days_employed()가 아직 main에 없음)"
assert not (df["DAYS_EMPLOYED"] == CODE).any(), "365243이 남아 있어요"

df["AGE"] = -df["DAYS_BIRTH"] / 365            # 나이 (년)
df["EMP_YEARS"] = -df["DAYS_EMPLOYED"] / 365   # 재직 연수 (년), 365243은 NaN
print("OK", raw.shape, "|", how)
# 이후 분석은 전부 df(365243 처리 후)로 한다.

---
## Part 0. 도입 — 끝에 몰린 막대 [예측, 실행, 해석]

1990년 미국 인구조사의 구역별 집값 데이터다 (sklearn 내장 California Housing, 처음 실행할 때만 인터넷에서 받음).
- `MedHouseVal`: 구역의 집값 중앙값 (단위 10만 달러)
- `HouseAge`: 구역의 주택 연식 중앙값 (년)

In [ ]:
# 확인할 것: 두 그래프 모두 오른쪽 끝 한 값에 막대가 높게 솟는지.
from sklearn.datasets import fetch_california_housing

cal = fetch_california_housing(as_frame=True).frame
fig, axes = plt.subplots(1, 2, figsize=(11, 3.8))
for ax, col, name in zip(axes, ["MedHouseVal", "HouseAge"], ["집값 중앙값 (10만 달러)", "주택 연식 중앙값 (년)"]):
    ax.hist(cal[col], bins=50, color=BLUE)
    ax.set(title=name, xlabel=col, ylabel="건수")
plt.tight_layout(); plt.show()

예측: 오른쪽 끝의 뾰족한 막대는 이상치인가? (예 / 아니오 / 모르겠다) → (    )

In [ ]:
# 확인할 것: 최댓값 하나에 수백 건이 몰려 있는지.
cal_cols = ["MedHouseVal", "HouseAge"]
cap = pd.DataFrame({
    "최댓값": cal[cal_cols].max(),
    "최댓값 건수": [(cal[c] == cal[c].max()).sum() for c in cal_cols],
})
cap["전체 대비"] = cap["최댓값 건수"] / len(cal)
cap

[해석] 이 값들은 이상치인가? 지우면 무엇이 사라지는가?

-

질문: Home Credit에도 튀어 보이지만 이상치가 아닌 값이 있을까? Part 2에서 다시 본다.

---
## Part 1. 데이터와 분포 확인 [실행, 해석]

- 이상치를 찾기 전에 데이터에 무엇이 들어 있고 어떤 모양인지만 확인. 이 Part는 실행만!
- 5주차에서 전체 122열을 봤으니, 여기서는 이번 주에 쓰는 열만 본다.

### 1-1. 데이터 개요

| 항목 | 내용 |
|---|---|
| 출처 | Kaggle Home Credit Default Risk — 소비자 대출 신청자 정보와 이후 상환 결과 |
| 단위 | 한 행 = 대출 신청 1건 (`SK_ID_CURR`) |
| 타깃 | `TARGET` 1 = 상환 어려움(연체), 0 = 그 외 |
| 시간 | 신청 날짜 없음. `DAYS_*`는 신청일 기준 며칠 전인지를 **음수**로 기록 |
| 표본 | 승인된 대출만 있음. 거절된 신청자는 타깃이 없음 |

### 1-2. 이번 주에 쓰는 변수 (`HomeCredit_columns_description.csv` 기준)

| 변수 | 설명 |
|---|---|
| `AMT_INCOME_TOTAL` | 연소득 |
| `AMT_CREDIT` | 대출액 |
| `AMT_ANNUITY` | 연간 상환액 |
| `AMT_GOODS_PRICE` | 대출로 사려는 상품 가격 |
| `CNT_CHILDREN` | 자녀 수 |
| `DAYS_BIRTH` | 신청일 기준 나이 (음수 일수) → `AGE` = −DAYS_BIRTH / 365 |
| `DAYS_EMPLOYED` | 현재 직장에서 일한 기간 (음수 일수) → `EMP_YEARS` = −DAYS_EMPLOYED / 365 |
| `NAME_INCOME_TYPE` | 소득 유형 (Working, Pensioner 등) |
| `OCCUPATION_TYPE` | 직업 |
| `NAME_EDUCATION_TYPE` | 학력 |

### 1-3. 데이터가 어떻게 생겼지?

In [ ]:
# 확인할 것: DAYS_*가 음수인지, 금액 단위가 어느 정도인지, 비어 있는 칸이 있는지.
use_cols = ["SK_ID_CURR", "TARGET", "AMT_INCOME_TOTAL", "AMT_CREDIT", "AMT_ANNUITY", "AMT_GOODS_PRICE",
            "CNT_CHILDREN", "DAYS_BIRTH", "DAYS_EMPLOYED", "NAME_INCOME_TYPE", "OCCUPATION_TYPE", "NAME_EDUCATION_TYPE"]
raw[use_cols].head()

In [ ]:
# 확인할 것: 결측이 있는 열, 숫자형인데 고유값이 적은 열(CNT_CHILDREN).
overview = pd.DataFrame({
    "dtype": raw[use_cols].dtypes.astype(str),
    "고유값 수": raw[use_cols].nunique(),
    "isnull 개수": raw[use_cols].isnull().sum(),
    "값 예시": [raw[c].unique()[:5].tolist() for c in use_cols],
})
overview

`DAYS_EMPLOYED`는 `isnull`이 0인데, 그러면 이 열에 결측이 없다고 할 수 있을까?? 1-5의 최댓값이랑 같이 확인

### 1-4. 타깃 분포

뒤에 나오는 모든 부도율을 이 기준선과 비교한다.

In [ ]:
# 확인할 것: 부도(1) 비율이 약 8.1%인지.
tgt = raw["TARGET"].value_counts().to_frame("건수")
tgt["비율"] = tgt["건수"] / len(raw)
tgt

### 1-5. 수치형 변수 요약

In [ ]:
# 확인할 것: mean과 50%의 차이(치우침), 75%와 max의 간격(극단값), DAYS_EMPLOYED에서 max만 양수인지.
cols6 = ["AMT_INCOME_TOTAL", "AMT_CREDIT", "AMT_ANNUITY", "CNT_CHILDREN", "DAYS_EMPLOYED", "DAYS_BIRTH"]
display(raw[cols6].describe().T)

amt_cols = ["AMT_INCOME_TOTAL", "AMT_CREDIT", "AMT_ANNUITY"]
(raw[amt_cols].max() / raw[amt_cols].quantile(0.75)).to_frame("max / 75% 값")

### 1-6. 금액 변수 분포 — 원래 스케일 vs 로그 스케일 (교재 10.5, 10.8)

`np.log1p(x)` = log(1 + x). 금액처럼 오른쪽 꼬리가 긴 변수의 모양을 볼 때 쓴다.

In [ ]:
# 확인할 것: 원래 스케일은 막대 하나와 납작한 상자, 로그 스케일은 봉우리와 상자 모양이 보이는지.
fig, axes = plt.subplots(len(amt_cols), 4, figsize=(17, 9))
for i, col in enumerate(amt_cols):
    s = df[col].dropna()
    log_s = np.log1p(s)
    axes[i, 0].hist(s, bins=50, color=BLUE)
    axes[i, 0].set(title=f"{col} 원래 스케일", ylabel="건수")
    sns.boxplot(x=s, ax=axes[i, 1], color=BLUE, fliersize=2)
    axes[i, 1].set_title("박스플롯 (원래)")
    axes[i, 2].hist(log_s, bins=50, color=ORANGE)
    axes[i, 2].set(title=f"{col} 로그 스케일", ylabel="건수")
    sns.boxplot(x=log_s, ax=axes[i, 3], color=ORANGE, fliersize=2)
    axes[i, 3].set_title("박스플롯 (로그)")
    for ax in axes[i]:
        ax.set_xlabel("")
plt.tight_layout(); plt.show()

### 1-7. 구간 수를 바꾸면 모양이 달라진다 (교재 10.5)

교재 10.5: 히스토그램은 처음에 20개 정도로 촘촘하게 나눠 보고 구간을 조금씩 줄여 간다. 너무 많으면 읽기 어렵고, 너무 적으면 정보가 사라진다.

In [ ]:
# 확인할 것: 구간 10개에서는 안 보이던 들쭉날쭉한 막대가 구간 100개에서 보이는지, 그게 무엇 때문인지.
log_inc = np.log1p(df["AMT_INCOME_TOTAL"])
fig, axes = plt.subplots(1, 3, figsize=(16, 3.6))
for ax, b in zip(axes, [10, 30, 100]):
    ax.hist(log_inc, bins=b, color=ORANGE)
    ax.set(title=f"로그 소득, 구간 {b}개", ylabel="건수")
plt.tight_layout(); plt.show()
print("가장 흔한 소득 5개:", df["AMT_INCOME_TOTAL"].value_counts().head(5).to_dict())

### 1-8. 나이, 재직 연수, 자녀 수

In [ ]:
# 확인할 것: 나이는 넓게 퍼져 있고, 재직 연수는 오른쪽 꼬리, 자녀 수는 0~2에 몰리고 끝에 드문 값이 있는지.
fig, axes = plt.subplots(2, 3, figsize=(16, 6))
for j, (col, name) in enumerate([("AGE", "나이 (년)"), ("EMP_YEARS", "재직 연수 (년, 365243 제외)")]):
    s = df[col].dropna()
    axes[0, j].hist(s, bins=50, color=BLUE)
    axes[0, j].set(title=name, ylabel="건수")
    sns.boxplot(x=s, ax=axes[1, j], color=BLUE, fliersize=2)

kids = df["CNT_CHILDREN"].value_counts().sort_index()
axes[0, 2].bar(kids.index.astype(str), kids.values, color=BLUE)
axes[0, 2].set_yscale("log")
axes[0, 2].set(title="자녀 수별 건수 (세로축 로그)", ylabel="건수")
sns.boxplot(x=df["CNT_CHILDREN"], ax=axes[1, 2], color=BLUE, fliersize=2)
plt.tight_layout(); plt.show()
print("자녀 수별 건수:", kids.to_dict())

### 1-9. 부도 여부별 소득 (교재 10.8.1)

그룹별 박스플롯에 평균(삼각형)을 같이 표시한다. 소득은 로그 스케일로 그린다.

In [ ]:
# 확인할 것: 부도자(1)의 소득 중앙값이 정상 상환자(0)보다 낮은지, 평균과 중앙값의 차이가 어느 쪽에서 큰지.
plot_df = df.assign(
    로그소득=np.log1p(df["AMT_INCOME_TOTAL"]),
    상환결과=df["TARGET"].map({0: "정상 상환 (0)", 1: "부도 (1)"}),
)
fig, ax = plt.subplots(figsize=(7, 4))
sns.boxplot(data=plot_df, x="상환결과", y="로그소득", order=["정상 상환 (0)", "부도 (1)"], showmeans=True,
            color=BLUE, fliersize=2, meanprops={"marker": "^", "markerfacecolor": ORANGE, "markeredgecolor": ORANGE}, ax=ax)
ax.set_title("부도 여부별 소득 (log1p)")
plt.tight_layout(); plt.show()

df.groupby("TARGET")["AMT_INCOME_TOTAL"].agg(["size", "mean", "median"])

### 1-10. 1차 메모 [해석] (2~3줄)

뒤에서 조심해야 할 것 같은 점만 적기. 뒤 Part에서 이 메모가 맞았는지 확인해보자.

-

---
## Part 2. 코드값을 먼저 처리한다.

5주차에서 본 것처럼 `DAYS_EMPLOYED = 365243`(약 1,000년)은 숫자처럼 보이지만 실제 근무 기간이 아니라 코드값이다. 전체의 18%(55,374건)다.

이번에는 이 값을 처리하지 **않은** 원본(`raw`)에 이상치 기준을 바로 쓰면 어떻게 되는지 본다.
- IQR 기준: Q1 − 1.5·IQR보다 작거나 Q3 + 1.5·IQR보다 크면 이상치 (박스플롯 수염 밖, 교재 10.8)
- z>3 기준: 평균에서 표준편차의 3배 넘게 떨어지면 이상치 (교재 11.2)

예측: 원본에 두 기준을 쓰면 365243이 이상치로 잡힐까?

| 기준 | 잡힐까? |
|---|---|
| IQR |  |
| z>3 |  |

In [ ]:
# 확인할 것: IQR은 365243을 전부 잡는데 z>3은 0개인지, z>3 경계가 실제 범위를 다 덮는지.
s = raw["DAYS_EMPLOYED"]
is_code = s == CODE
q1, q3 = s.quantile([0.25, 0.75])
iqr = q3 - q1
iqr_out = (s < q1 - 1.5 * iqr) | (s > q3 + 1.5 * iqr)
z_out = ((s - s.mean()) / s.std()).abs() > 3

print("실제 범위:", s.min(), "~", s.max())
pd.DataFrame({
    "평균": [s.mean()],
    "표준편차": [s.std()],
    "z>3 하한": [s.mean() - 3 * s.std()],
    "z>3 상한": [s.mean() + 3 * s.std()],
    "IQR 이상치": [iqr_out.sum()],
    "그중 365243": [(iqr_out & is_code).sum()],
    "z>3 이상치": [z_out.sum()],
}, index=["원본 DAYS_EMPLOYED"]).T

In [ ]:
# 확인할 것: 365243 행이 거의 전부 연금 수령자인지, 부도율이 나머지와 다른지.
display(raw.loc[is_code, "NAME_INCOME_TYPE"].value_counts().to_frame("365243 행의 소득 유형"))
raw.groupby(is_code.map({True: "365243", False: "그 외"}))["TARGET"].agg(["size", "mean"]).rename(columns={"size": "건수", "mean": "부도율"})

### 2-1. 처리한 뒤 다시 세기 [빈칸]

`df`의 `EMP_YEARS`(365243 처리 후, 년 단위)에 같은 두 기준을 쓴다.

In [ ]:
# 확인할 것: 365243을 처리한 뒤에는 z>3도 이상치를 잡기 시작하는지.
s2 = df["EMP_YEARS"].dropna()
q1_2, q3_2 = s2.quantile([0.25, 0.75])
iqr_2 = q3_2 - q1_2
pd.Series({
    "IQR 이상치": ((s2 < q1_2 - 1.5 * iqr_2) | (s2 > q3_2 + 1.5 * iqr_2)).sum(),
    "z>3 이상치": TODO,   # 위 셀과 같은 z>3 기준을 s2에 적용한 개수
}, name="처리 후 EMP_YEARS").to_frame()

[해석] 원본에서 z>3이 0개인 이유는? (평균, 표준편차, z>3 경계를 실제 범위와 비교)

-

[해석] 365243을 "이상치"로 보고 지우면 무엇을 잃는가?

-

[해석] 교재 11.2는 이상치를 지우는 대신 변수로 만들어 설명력을 높이는 방법을 소개한다(전문직/사무직 예시). 365243에는 이 방법을 어떻게 쓸 수 있는가?

-

---
## Part 3. 기준이 다르면 이상치 개수가 다르다 [예측, 빈칸, 해석]

교재 10.8과 11.2에 나온 기준 네 가지를 같은 변수에 써 본다.

| 기준 | 정의 | 교재 |
|---|---|---|
| IQR×1.5 | Q1 − 1.5·IQR 미만 또는 Q3 + 1.5·IQR 초과 (박스플롯 수염 밖) | 10.8 |
| IQR×3 | Q1 − 3·IQR 미만 또는 Q3 + 3·IQR 초과 | 11.2.1 실습 |
| z>3 | (x − 평균)의 절댓값 / 표준편차 > 3 | 11.2 |
| MAD z>3.5 | 0.6745 × (x − 중앙값)의 절댓값 / MAD > 3.5. MAD는 (x − 중앙값) 절댓값들의 중앙값 | 11.2 (MAD), 기준값 3.5는 Iglewicz·Hoaglin(1993) |

### 3-1. 정규분포라면 각 기준은 몇 σ인가 [실행]

교재 그림 10.26은 IQR×1.5 경계를 ±2.698σ로 그린다. 같은 계산으로 IQR×3 경계도 구한다.

In [ ]:
# 확인할 것: IQR×1.5 경계가 ±2.698σ, IQR×3 경계가 ±4.72σ인지, 정규분포에서 그 밖에 놓이는 비율.
q3_sigma = stats.norm.ppf(0.75)   # 표준정규분포의 Q3 (0.6745σ)
iqr_sigma = 2 * q3_sigma          # 표준정규분포의 IQR (1.349σ)
rows = []
for name, edge in [("IQR×1.5", q3_sigma + 1.5 * iqr_sigma), ("IQR×3", q3_sigma + 3 * iqr_sigma), ("z>3", 3.0)]:
    rows.append((name, round(edge, 3), f"{2 * stats.norm.sf(edge):.5%}"))
pd.DataFrame(rows, columns=["기준", "경계 (±σ)", "정규분포에서 밖의 비율"]).set_index("기준")

체크: 교재 p.223은 IQR×3을 "약 5σ", p.199는 박스플롯 수염 끝을 "평균 좌우 3표준편차"라고 했다. 계산하면 각각 ±4.72σ, ±2.698σ다.

### 3-2. 변수 × 기준 이상치 개수 [빈칸]

`outlier_rules()`는 Part 4·5에서도 쓴다.

예측: 소득(`AMT_INCOME_TOTAL`)에서 이상치가 가장 많이 잡히는 기준 → (    ), 가장 적게 잡히는 기준 → (    )

In [ ]:
# 확인할 것: 같은 변수인데 기준마다 개수가 몇 배씩 다른지, 자녀 수의 MAD가 0이라 계산이 안 되는지.
def outlier_rules(s):
    """네 기준으로 이상치 여부(True/False)를 돌려준다. 결측은 빼고 계산한다."""
    s = s.dropna()
    q1, q3 = s.quantile([0.25, 0.75])
    iqr = q3 - q1
    med = s.median()
    mad = (s - med).abs().median()
    return pd.DataFrame({
        "IQR×1.5": TODO,   # Q1 - 1.5*IQR 미만 또는 Q3 + 1.5*IQR 초과
        "IQR×3": (s < q1 - 3 * iqr) | (s > q3 + 3 * iqr),
        "z>3": TODO,       # (x - 평균)의 절댓값 / 표준편차 > 3
        "MAD z>3.5": (0.6745 * (s - med).abs() / mad > 3.5) if mad > 0 else np.nan,   # MAD가 0이면 계산 불가
    })


check_cols = ["AMT_INCOME_TOTAL", "AMT_CREDIT", "AMT_ANNUITY", "CNT_CHILDREN", "EMP_YEARS", "AGE"]
counts = pd.DataFrame({c: outlier_rules(df[c]).sum(min_count=1) for c in check_cols}).T
counts.astype("Int64")

### 3-3. 로그 스케일에서 다시 세기 [실행]

In [ ]:
# 확인할 것: 금액 변수를 로그로 바꾸면 IQR 이상치 개수가 얼마나 줄어드는지.
log_counts = pd.DataFrame({c: outlier_rules(np.log1p(df[c])).sum(min_count=1) for c in amt_cols}).T
pd.DataFrame({
    "원래 스케일 IQR×1.5": counts.loc[amt_cols, "IQR×1.5"],
    "로그 스케일 IQR×1.5": log_counts["IQR×1.5"],
}).astype("Int64")

[해석] 소득에서 z>3이 IQR×1.5보다 훨씬 적게 잡힌 이유는?

-

[해석] 자녀 수에 IQR을 쓰면 무엇이 문제인가?

-

[해석] 그럼 어떤 기준을 써야 하는가?

-

---
## Part 4. 지우기 전에 행을 열어본다 [예측, 빈칸, 해석]

교재 11.2: 고객의 연령이 225세로 입력되어 있으면 잘못된 데이터인 것을 바로 알 수 있다. 통계 기준이 아니라 다른 열과의 논리로 오류를 판단하는 것.

예측: 소득 1위(117,000,000) 신청자는 진짜일까? (진짜 / 오류 / 모르겠다) → (    )

### 4-1. 소득 상위 행 열기 [빈칸]

In [ ]:
# 확인할 것: 1위의 대출액·상환액·직업이 그 소득과 어울리는지, 1위와 2위의 간격.
show_cols = ["SK_ID_CURR", "TARGET", "AMT_INCOME_TOTAL", "AMT_CREDIT", "AMT_ANNUITY", "AMT_GOODS_PRICE",
             "NAME_INCOME_TYPE", "OCCUPATION_TYPE", "NAME_EDUCATION_TYPE", "AGE", "EMP_YEARS"]
top = TODO   # df에서 AMT_INCOME_TOTAL 상위 6행의 show_cols
top["상환액 / 소득"] = top["AMT_ANNUITY"] / top["AMT_INCOME_TOTAL"]
display(top)

print(f"1위 / 2위 소득: {top['AMT_INCOME_TOTAL'].iloc[0] / top['AMT_INCOME_TOTAL'].iloc[1]:.1f}배")
print(f"전체 상환액 / 소득 중앙값: {(df['AMT_ANNUITY'] / df['AMT_INCOME_TOTAL']).median():.1%}")
laborers = df.loc[df["OCCUPATION_TYPE"] == "Laborers", "AMT_INCOME_TOTAL"]
print(f"Laborers 소득 중앙값 {laborers.median():,.0f} / 99% 값 {laborers.quantile(0.99):,.0f}")

[해석] 1위는 오류인가, 진짜 극단값인가? 2~6위는? 데이터만으로 확정할 수 있는가?

-

### 4-2. 비율로 보면 튀는 행 [심화, 실행]

원 변수에서는 평범한데 두 변수의 비율에서 튀는 행이 있다. 연간 상환액이 소득보다 큰 신청자, 대출액/소득 상위 1%를 본다.

In [ ]:
# 확인할 것: 상환액 / 소득 > 1인 행이 몇 건인지, 분모(소득)가 작은지, 소득 유형과 부도율.
df["ANNUITY_INCOME"] = df["AMT_ANNUITY"] / df["AMT_INCOME_TOTAL"]
over = df["ANNUITY_INCOME"] > 1
inc_flag = outlier_rules(df["AMT_INCOME_TOTAL"])["IQR×1.5"].reindex(df.index, fill_value=False)
ann_flag = outlier_rules(df["AMT_ANNUITY"])["IQR×1.5"].reindex(df.index, fill_value=False)

print(f"상환액 > 소득: {over.sum()}건, 부도율 {df.loc[over, 'TARGET'].mean():.1%} (전체 {df['TARGET'].mean():.1%})")
print(f"그중 소득이 IQR 이상치: {(over & inc_flag).sum()}건 / 상환액이 IQR 이상치: {(over & ann_flag).sum()}건")
display(df.loc[over, "NAME_INCOME_TYPE"].value_counts().to_frame("소득 유형"))
display(df.loc[over, ["AMT_INCOME_TOTAL", "AMT_ANNUITY", "AMT_CREDIT", "ANNUITY_INCOME", "NAME_INCOME_TYPE", "TARGET"]]
        .sort_values("ANNUITY_INCOME", ascending=False).head(6))

# 대출액 / 소득: 상위 1%
df["CREDIT_INCOME"] = df["AMT_CREDIT"] / df["AMT_INCOME_TOTAL"]
cr_cut = df["CREDIT_INCOME"].quantile(0.99)
cr_hi = df["CREDIT_INCOME"] > cr_cut
cr_flag = outlier_rules(df["AMT_CREDIT"])["IQR×1.5"].reindex(df.index, fill_value=False)
print(f"대출액 / 소득 상위 1% (> {cr_cut:.1f}배): {cr_hi.sum():,}건, 부도율 {df.loc[cr_hi, 'TARGET'].mean():.1%}")
print(f"그중 소득·대출액 어느 쪽도 IQR 이상치가 아닌 행: {(cr_hi & ~inc_flag & ~cr_flag).sum():,}건")
print(f"이 그룹 소득 중앙값 {df.loc[cr_hi, 'AMT_INCOME_TOTAL'].median():,.0f} (전체 {df['AMT_INCOME_TOTAL'].median():,.0f})")

[해석] 이 행들은 오류인가? 비율 변수의 극단값은 어디부터 확인해야 하는가?

-

---
## Part 5. 지우면 결론이 바뀐다 [예측, 빈칸, 해석]
6개 변수 중 **하나라도** IQR×1.5 이상치인 행을 전부 지우고, 남은 데이터로 부도율과 소득을 다시 계산한다.

### 5-0. 예측 (코드 실행 전)

| | 예측 |
|---|---|
| 삭제되는 행 비율 |  |
| 부도율 (현재 8.07%) | 올라감 / 내려감 / 그대로 |
| 소득 중앙값 (현재 147,150) | 올라감 / 내려감 / 그대로 |

### 5-1. 일괄 삭제 전후 [빈칸]

In [ ]:
# 확인할 것: 몇 %가 사라지는지, 부도율·소득 평균·소득 중앙값이 어느 쪽으로 움직이는지.
flags = pd.DataFrame({c: outlier_rules(df[c])["IQR×1.5"].reindex(df.index, fill_value=False) for c in check_cols})
any_out = TODO   # flags에서 한 열이라도 True인 행
kept = df[~any_out]


def report(d):
    return [len(d), d["TARGET"].mean(), d["AMT_INCOME_TOTAL"].mean(), d["AMT_INCOME_TOTAL"].median()]


print(f"삭제 비율: {any_out.mean():.1%} ({any_out.sum():,}행)")
pd.DataFrame({"전체": report(df), "일괄 삭제 후": report(kept)}, index=["행 수", "부도율", "소득 평균", "소득 중앙값"])

### 5-2. 지운 행은 어떤 사람들인가 [실행]

In [ ]:
# 확인할 것: 변수별 이상치 그룹의 부도율이 전체 기준선보다 높은지 낮은지.
base = df["TARGET"].mean()
rate = pd.DataFrame({
    "이상치 건수": flags.sum(),
    "이상치 부도율": [df.loc[flags[c], "TARGET"].mean() for c in flags],
    "나머지 부도율": [df.loc[~flags[c], "TARGET"].mean() for c in flags],
})
display(rate)

plot_rate = rate.dropna()
fig, ax = plt.subplots(figsize=(8, 3.5))
ax.barh(plot_rate.index, plot_rate["이상치 부도율"], color=ORANGE)
ax.axvline(base, color=GRAY, ls="--", label=f"전체 부도율 {base:.1%}")
ax.set(xlabel="부도율", title="IQR 이상치 그룹의 부도율")
ax.invert_yaxis()
ax.legend(frameon=False, loc="lower right")
plt.tight_layout(); plt.show()

[해석] 부도율은 왜 움직였는가? 팀장에게 "고소득자 빼고 다시 계산했습니다"라고만 보고하면 정직한가?

-

### 5-3. 소득 처리 방식 비교 [심화, 빈칸]

지우는 것 말고도 방법이 있다 (교재 11.2: 제거 / 상·하한 대체 / 가중치 조정). 소득 하나만 네 방식으로 처리하고 정상 상환자와 부도자의 분위수를 비교한다.

In [ ]:
# 확인할 것: 어떤 방식에서 "부도자 소득이 더 낮다"는 차이가 줄거나 사라지는지.
inc = df["AMT_INCOME_TOTAL"]
p01, p99 = inc.quantile([0.01, 0.99])
inc_out = outlier_rules(inc)["IQR×1.5"]
versions = {
    "그대로": (df["TARGET"], inc),
    "IQR 제거": (df.loc[~inc_out, "TARGET"], inc[~inc_out]),
    "1·99% 상한": (df["TARGET"], TODO),   # inc를 p01 ~ p99 범위로 자르기 (clip)
}
rows = []
for name, (t, x) in versions.items():
    for target in [0, 1]:
        xs = x[t == target]
        rows.append((name, "정상 (0)" if target == 0 else "부도 (1)", len(xs), xs.mean(), xs.quantile(0.25), xs.median(), xs.quantile(0.75)))
# 로그 변환: 로그 스케일에서 평균을 낸 뒤 원래 단위로 되돌린다 (기하평균). 순서는 그대로라 분위수는 같다.
for target in [0, 1]:
    xs = inc[df["TARGET"] == target]
    rows.append(("로그 평균 → 원래 단위", "정상 (0)" if target == 0 else "부도 (1)", len(xs),
                 np.expm1(np.log1p(xs).mean()), xs.quantile(0.25), xs.median(), xs.quantile(0.75)))
display(pd.DataFrame(rows, columns=["방식", "그룹", "건수", "평균", "25%", "중앙값", "75%"]).set_index(["방식", "그룹"]))

print(f"상한 경계: {p01:,.0f} ~ {p99:,.0f}")
d1 = inc[df["TARGET"] == 1]
print(f"부도 그룹 평균: 전체 {d1.mean():,.0f} / 소득 1위 1행 제외 {d1.drop(d1.idxmax()).mean():,.0f}")

[해석] 어느 방식이 두 그룹의 차이를 지웠는가?

-

[해석] 상한 처리 후 평균 차이는 왜 오히려 커졌는가?

-

[해석] 팀장에게 보고할 때는 어떤 방식을 쓰겠는가?

-

---
## Part 6. 분류표 완성, AI와 비교 [빈칸, 해석]

### 6-1. 오늘 본 값 분류 [빈칸]

분류: 오류 / 코드값 / 절단 / 진짜 극단값 / 통계적으로만 드문 값 (판단 보류도 가능). 근거에는 Part 번호나 숫자를 적는다.

In [ ]:
# 확인할 것: 행마다 분류와 근거(숫자 또는 다른 열과의 관계)를 적었는지.
outlier_types = pd.DataFrame([
    # 값,                                   분류,     근거
    ("DAYS_EMPLOYED = 365243",            "TODO",   "TODO"),
    ("MedHouseVal = 5.00001",             "TODO",   "TODO"),
    ("AMT_INCOME_TOTAL = 117,000,000",    "TODO",   "TODO"),
    ("AMT_INCOME_TOTAL 3·4·6위",           "TODO",   "TODO"),
    ("CNT_CHILDREN 3~4명",                 "TODO",   "TODO"),
    ("상환액 > 소득 33건",                  "TODO",   "TODO"),
], columns=["값", "분류", "근거"])
outlier_types

### 6-2. AI와 비교 [빈칸, 해석]

내 분석을 끝낸 뒤, AI에게 아래 프롬프트를 그대로 넣고 결과를 점검한다.

> "IQR 방식으로 이상치를 제거해줘."

| 점검 항목 | 통과 / 실패 | 근거 (AI 답변의 어느 부분) |
|---|---|---|
| 365243을 이상치로 보고 지우지 않았는가 |  |  |
| 모든 수치 열에 IQR을 일괄 적용하지 않았는가 (자녀 수, ID, 플래그 열) |  |  |
| 꼬리가 긴 변수의 분포(로그 스케일)를 먼저 확인했는가 |  |  |
| 지우기 전에 극단값 행을 열어봤는가 |  |  |
| 제거 후 남은 행 수와 부도율을 보고했는가 |  |  |

[해석] AI와 내가 달랐던 점, 누가 맞았는지

-

### 토론 [해석]

① 팀장 말대로 "고소득자 몇 명"을 빼면 부도율·평균 소득 보고는 어떻게 바뀌고, 그게 정직한 보고일까?

-

② 오류인지 진짜 극단값인지 데이터만으로 판단할 수 없을 때는 어떻게 할까?

-

---
## 마무리 [해석]

오늘의 결론 세 줄. 결론을 바꾼 세 가지 선택을 오늘 본 숫자로 적는다.

1. 코드값을 먼저 처리하느냐 →
2. 어떤 기준으로 세느냐 →
3. 지우느냐 남기느냐 →

PR 전 체크리스트 (이번 주 추가 항목)
- [ ] 코드값(365243)을 처리한 뒤 이상치를 봤는가
- [ ] 기준을 고른 이유를 적었는가 (이산형 변수에 IQR을 쓰지 않았는가)
- [ ] 지우기 전에 해당 행을 직접 열어봤는가
- [ ] 이상치 제거가 타깃 분포(부도율)를 바꾸지 않았는지 확인했는가
- [ ] 평균과 중앙값을 함께 보고했는가 (D-003)
- [ ] 핵심 결과 한 줄이 노트북의 숫자와 일치하는가
- [ ] 노트북 출력 제거, 본인 폴더만 수정, 데이터 파일 없음